This Notebook uses constructed data offline. Fixed filing facts and reference figures are historical teaching materials; new calculations do not reproduce a market strategy.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). See this guide for run steps, data requirements, and expected outputs.

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch09_optimization_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# Chapter 9: convex optimization

All inputs are constructed; there are no market observations or backtests. The holding period is one year, expected simple returns are decimals, and covariance covers that same period. `risk_aversion=1` has inverse-return units and must be rescaled if returns are expressed in percentage points. The shared interface is `src/quant_course/optimization_tools.py`. The direct least-squares and production LP examples expose their variables, objectives and constraints. No release, signal, trade or realized-return clock is specified.

In [ ]:
from pathlib import Path
import sys

import cvxpy as cp
import numpy as np

ROOT = Path.cwd().resolve()
assert (ROOT / "src" / "quant_course").exists(), "Run this notebook from the quant_course project root"
sys.path.insert(0, str(ROOT / "src"))
from quant_course.optimization_tools import OptimizationError, solve_mean_variance

print("interpreter:", sys.executable)
print("CVXPY:", cp.__version__)
print("installed solvers:", cp.installed_solvers())
assert sys.prefix != sys.base_prefix, "Use the isolated reader environment"

## Two-asset mean–variance quadratic program

Choose fully invested, long-only weights. The objective trades expected return against a variance penalty using hand-checkable annual constructed inputs. The code checks weights (5/11, 6/11) and minimum-variance weights (8/11, 3/11).

In [ ]:
mu = np.array([0.05, 0.08])
sigma = np.array([[0.04, 0.01], [0.01, 0.09]])
risk_aversion = 1.0

portfolio = solve_mean_variance(mu, sigma, risk_aversion=risk_aversion)
minimum_variance = solve_mean_variance([0.0, 0.0], sigma, risk_aversion=1.0)
print("status / solver:", portfolio.solver_status, "/", portfolio.solver_name)
print("mean-variance weights:", portfolio.weights)
print("annual expected return:", portfolio.expected_return)
print("annual variance:", portfolio.variance)
print("objective:", portfolio.objective)
print("minimum-variance weights:", minimum_variance.weights)
print("minimum annual variance:", minimum_variance.variance)
np.testing.assert_allclose(portfolio.weights, [5 / 11, 6 / 11], atol=1e-7, rtol=0)
np.testing.assert_allclose(minimum_variance.weights, [8 / 11, 3 / 11], atol=1e-7, rtol=0)

## Gradient and optimality diagnostics

Only the budget equality binds at this interior solution, so the two objective-gradient entries agree. The interface returns primal-feasibility, stationarity and complementary-slackness residuals. Only primal feasibility has a default threshold check; dual signs and the duality gap are not certified. These diagnostics are not a complete numerical optimality certificate. The next cell also checks infeasibility when both weights are capped at 0.4.

In [ ]:
gradient = risk_aversion * sigma @ portfolio.weights - mu
print("gradient:", gradient)
print("gradient difference:", gradient[0] - gradient[1])
print("budget residual:", portfolio.budget_residual)
print("stationarity residual:", portfolio.stationarity_residual)
print("complementarity residual:", portfolio.complementarity_residual)
print("budget dual:", portfolio.budget_dual)
np.testing.assert_allclose(gradient[0], gradient[1], atol=1e-7, rtol=0)
assert portfolio.stationarity_residual < 1e-6

In [ ]:
try:
    solve_mean_variance(mu, sigma, max_weight=0.4, solver="CLARABEL")
except OptimizationError as exc:
    print("infeasible-cap result:", exc.solver_status, str(exc))
    assert exc.solver_status == "infeasible"
else:
    raise AssertionError("a 0.4 cap on both of two assets must be infeasible")

## Least squares

Regress four constructed responses on an intercept and one predictor. NumPy and CVXPY minimize the same residual sum of squares. Compare coefficients (0.9, 1.4) and SSE = 0.2. Expressing an optimization problem does not supply regression inference.

In [ ]:
x = np.array([0.0, 1.0, 2.0, 3.0])
y = np.array([1.0, 2.0, 4.0, 5.0])
A = np.column_stack([np.ones_like(x), x])

beta_np, *_ = np.linalg.lstsq(A, y, rcond=None)
beta = cp.Variable(2, name="intercept_and_slope")
ols_problem = cp.Problem(cp.Minimize(cp.sum_squares(A @ beta - y)))
ols_problem.solve()
print("[intercept, slope]:", beta.value)
print("SSE:", ols_problem.value)
np.testing.assert_allclose(beta.value, [0.9, 1.4], atol=1e-7, rtol=0)
np.testing.assert_allclose(beta.value, beta_np, atol=1e-7, rtol=0)
np.testing.assert_allclose(ols_problem.value, 0.2, atol=1e-8, rtol=0)

## A linear program with units

Variables count production lots per day. Contributions are 3 and 2 thousand EUR per lot; available machine and labor time are 100 and 80 hours per day. Constraints specify hours per lot. The constructed optimum of (40, 20) lots per day gives EUR 160 thousand per day. Dual values describe the local marginal contribution of an extra daily resource hour; they cannot be extrapolated through changes in the active constraints.

In [ ]:
production = cp.Variable(2, nonneg=True, name="lots_per_day")
machine_limit = 2 * production[0] + production[1] <= 100  # machine-hours/day
labor_limit = production[0] + 2 * production[1] <= 80     # labor-hours/day
lp = cp.Problem(
    cp.Maximize(np.array([3.0, 2.0]) @ production),  # kEUR/day
    [machine_limit, labor_limit],
)
lp.solve(solver="CLARABEL")
print("lots/day [A, B]:", production.value)
print("contribution (kEUR/day):", lp.value)
print("binding resource use [machine h/day, labor h/day]:",
      2 * production.value[0] + production.value[1],
      production.value[0] + 2 * production.value[1])
assert lp.status == cp.OPTIMAL
np.testing.assert_allclose(production.value, [40.0, 20.0], atol=1e-6, rtol=0)
np.testing.assert_allclose(lp.value, 160.0, atol=1e-6, rtol=0)

## Objective geometry

The frozen reference plot varies the weight of asset 1 from 0 to 1, with the weight of asset 2 equal to 1−w1. Its vertical axis is the annual risk–return objective. The marked interior solution illustrates curvature, not historical performance or an empirical efficient frontier. This cell displays an existing PNG rather than saving a new plot.

In [ ]:
from IPython.display import Image, display
figure_path = ROOT / "figures" / "a2" / "ch09_objective_curve.png"
display(Image(filename=str(figure_path)))